In [1]:
import pandas as pd 
import numpy as np

### Load data

In [2]:
df_prices = pd.read_csv("/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/Data/smard/Gro_handelspreise_201810010000_202412010000_Stunde.csv", sep=";")
df_prod = pd.read_csv("/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/Data/smard/Realisierte_Erzeugung_201810010000_202412010000_Stunde.csv", sep=";")
df_cons = pd.read_csv("/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/Data/smard/Realisierter_Stromverbrauch_201810010000_202412010000_Stunde.csv", sep=";")
df_prod_forecast = pd.read_csv("/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/Data/smard/Prognostizierte_Erzeugung_Day-Ahead_201810010000_202412010000_Stunde.csv", sep=";")
df_cons_forecast = pd.read_csv("/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/Data/smard/Prognostizierter_Stromverbrauch_201810010000_202412010000_Stunde.csv", sep=";")

### Preprocessing 

- Adjusting dtypes 
- Dropping values before 2019-01-01 because of NaN-values in forecast data
- Select only needed columns of dfs
- Fill remaining NaN values with values from the previous day or day after
- Close missing time steps with values from the previous day or the day after  
- Merge dfs and do a split into 3 datasets (forecast, genmodel / policy training, policy testing)

In [3]:
# Helper function to convert strings that contain floats in german float format to float
def convert_floats(
    df: pd.DataFrame,
    column_names: list[str]
) -> pd.DataFrame:
    
    for column_name in column_names:

        if df[column_name].dtype == object:
            df[column_name] = df[column_name].replace("-", np.nan)
            df[column_name] = df[column_name].str.replace(".", "").str.replace(",", ".").astype(float)

        else:
            continue

    return df

In [5]:
# Adjust data types
df_prices["Datum von"] = pd.to_datetime(df_prices["Datum von"], format="%d.%m.%Y %H:%M")
df_prod["Datum von"] = pd.to_datetime(df_prod["Datum von"], format="%d.%m.%Y %H:%M")
df_cons["Datum von"] = pd.to_datetime(df_cons["Datum von"], format="%d.%m.%Y %H:%M")
df_prod_forecast["Datum von"] = pd.to_datetime(df_prod_forecast["Datum von"], format="%d.%m.%Y %H:%M")
df_cons_forecast["Datum von"] = pd.to_datetime(df_cons_forecast["Datum von"], format="%d.%m.%Y %H:%M")

df_prices = convert_floats(df_prices, df_prices.columns[2:].to_list())
df_prod = convert_floats(df_prod, df_prod.columns[2:].to_list())
df_cons = convert_floats(df_cons, df_cons.columns[2:].to_list())
df_prod_forecast = convert_floats(df_prod_forecast, df_prod_forecast.columns[2:].to_list())
df_cons_forecast = convert_floats(df_cons_forecast, df_cons_forecast.columns[2:].to_list())

In [6]:
df_prod_forecast[df_prod_forecast.isna().any(axis=1)]

,Datum von,Datum bis,Gesamt [MWh] Originalauflösungen,Photovoltaik und Wind [MWh] Berechnete Auflösungen,Wind Offshore [MWh] Berechnete Auflösungen,Wind Onshore [MWh] Berechnete Auflösungen,Photovoltaik [MWh] Berechnete Auflösungen,Sonstige [MWh] Originalauflösungen
37009,2022-12-21 00:00:00,21.12.2022 01:00,NaN,19428.00,2412.00,17016.00,0.0,NaN
37010,2022-12-21 01:00:00,21.12.2022 02:00,NaN,18152.00,2218.75,15933.25,0.0,NaN
37011,2022-12-21 02:00:00,21.12.2022 03:00,NaN,17155.25,1977.25,15178.00,0.0,NaN
37012,2022-12-21 03:00:00,21.12.2022 04:00,NaN,16434.25,1865.25,14569.00,0.0,NaN
37013,2022-12-21 04:00:00,21.12.2022 05:00,NaN,15736.50,1806.50,13930.00,0.0,NaN
...,...,...,...,...,...,...,...,...
53252,2024-10-27 19:00:00,27.10.2024 20:00,NaN,5714.50,1637.00,4077.50,0.0,NaN
53253,2024-10-27 20:00:00,27.10.2024 21:00,NaN,4942.00,1652.25,3289.75,0.0,NaN
53254,2024-10-27 21:00:00,27.10.2024 22:00,NaN,4739.00,1852.75,2886.25,0.0,NaN
53255,2024-10-27 22:00:00,27.10.2024 23:00,NaN,5042.25,2242.00,2800.25,0.0,NaN


In [7]:
df_cons_forecast[df_cons_forecast.isna().any(axis=1)]

,Datum von,Datum bis,Gesamt (Netzlast) [MWh] Berechnete Auflösungen,Residuallast [MWh] Berechnete Auflösungen
0,2018-10-01 00:00:00,01.10.2018 01:00,NaN,NaN
1,2018-10-01 01:00:00,01.10.2018 02:00,NaN,NaN
26,2018-10-02 02:00:00,02.10.2018 03:00,NaN,NaN
27,2018-10-02 03:00:00,02.10.2018 04:00,NaN,NaN
28,2018-10-02 04:00:00,02.10.2018 05:00,NaN,NaN
...,...,...,...,...
43101,2023-08-31 21:00:00,31.08.2023 22:00,NaN,NaN
43102,2023-08-31 22:00:00,31.08.2023 23:00,NaN,NaN
43103,2023-08-31 23:00:00,01.09.2023 00:00,NaN,NaN
44496,2023-10-29 00:00:00,29.10.2023 01:00,NaN,NaN


In [8]:
# Drop before 2019
start_date = "2019-01-01"

df_prices = df_prices[df_prices["Datum von"] >= start_date].reset_index(drop=True)
df_prod = df_prod[df_prod["Datum von"] >= start_date].reset_index(drop=True)
df_cons = df_cons[df_cons["Datum von"] >= start_date].reset_index(drop=True)
df_prod_forecast = df_prod_forecast[df_prod_forecast["Datum von"] >= start_date].reset_index(drop=True)
df_cons_forecast = df_cons_forecast[df_cons_forecast["Datum von"] >= start_date].reset_index(drop=True)

In [9]:
# Select relevant columns and rename them 
df_prices = df_prices[["Datum von", "Deutschland/Luxemburg [€/MWh] Originalauflösungen", "Belgien [€/MWh] Originalauflösungen", "Dänemark 1 [€/MWh] Originalauflösungen", "Frankreich [€/MWh] Originalauflösungen", "Niederlande [€/MWh] Originalauflösungen", "Schweiz [€/MWh] Originalauflösungen", "Italien (Nord) [€/MWh] Originalauflösungen"]]
df_prices.columns = ["Date", "DAP DE/LU", "DAP BE", "DAP DK1", "DAP FR", "DAP NL", "DAP CH", "DAP IT-N"]

df_cons = df_cons[["Datum von", "Gesamt (Netzlast) [MWh] Berechnete Auflösungen"]]
df_cons.columns = ["Date", "Total Consumption"]

df_prod["Total production"] = df_prod.iloc[:, 2:].sum(axis=1)
df_prod["Reg production"] = df_prod.iloc[:, 2:8].sum(axis=1)
df_prod["Conv production"] = df_prod.iloc[:, 8:14].sum(axis=1)
df_prod = df_prod[["Datum von", "Total production", "Reg production", "Conv production"]]
df_prod.columns = ["Date", "Total Production", "Reg Production", "Conv Production"]

In [10]:
df_prod_forecast[df_prod_forecast.isna().any(axis=1)].count()

Datum von                                             122
Datum bis                                             122
Gesamt [MWh] Originalauflösungen                        0
Photovoltaik und Wind [MWh] Berechnete Auflösungen    120
Wind Offshore [MWh] Berechnete Auflösungen            122
Wind Onshore [MWh] Berechnete Auflösungen             120
Photovoltaik [MWh] Berechnete Auflösungen             120
Sonstige [MWh] Originalauflösungen                      0
dtype: int64

In [12]:
df_cons_forecast[df_cons_forecast.isna().any(axis=1)].count()

Datum von                                         194
Datum bis                                         194
Gesamt (Netzlast) [MWh] Berechnete Auflösungen      0
Residuallast [MWh] Berechnete Auflösungen           0
dtype: int64

In [13]:
# Helper function to fill missing values with the values from the previous or next day
def df_fill_na_with_previous_day(df):
    for col in df.columns:
        df[col] = df[col].ffill(limit=24).bfill(limit=24)
    return df

df_prod_forecast = df_fill_na_with_previous_day(df_prod_forecast)
df_cons_forecast = df_fill_na_with_previous_day(df_cons_forecast)


In [16]:
df_prod_forecast[["Datum von", "Photovoltaik [MWh] Berechnete Auflösungen"]].to_csv("../Data/solar_prod_forecast.csv")
df_cons_forecast[["Datum von", "Gesamt (Netzlast) [MWh] Berechnete Auflösungen"]].to_csv("../Data/total_cons_forecast.csv")

In [16]:
df_prod_forecast = df_prod_forecast[["Datum von", "Gesamt [MWh] Originalauflösungen"]]
df_prod_forecast.columns = ["Date", "Total production forecast"]

df_cons_forecast = df_cons_forecast[["Datum von", "Gesamt (Netzlast) [MWh] Berechnete Auflösungen", "Residuallast [MWh] Berechnete Auflösungen"]]
df_cons_forecast.columns = ["Date", "Total Consumption forecast", "Residual load forecast"]

In [20]:
df_prod

,Date,Total Production,Reg Production,Conv Production
0,2019-01-01 00:00:00,53501.25,30060.25,23441.00
1,2019-01-01 01:00:00,52817.25,31767.75,21049.50
2,2019-01-01 02:00:00,52169.50,32131.50,20038.00
3,2019-01-01 03:00:00,52805.25,33628.00,19177.25
4,2019-01-01 04:00:00,54352.75,35395.75,18957.00
...,...,...,...,...
51859,2024-11-30 19:00:00,53451.50,25936.25,27515.25
51860,2024-11-30 20:00:00,50973.75,24395.75,26578.00
51861,2024-11-30 21:00:00,48334.00,23067.50,25266.50
51862,2024-11-30 22:00:00,46959.50,21469.75,25489.75


In [17]:
df_prod_forecast[df_prod_forecast.isna().any(axis=1)].count()

Date                         0
Total production forecast    0
dtype: int64

In [18]:
df_cons_forecast[df_cons_forecast.isna().any(axis=1)].count()

Date                          0
Total Consumption forecast    0
Residual load forecast        0
dtype: int64

In [479]:
df_scenario1 = df_prices.merge(df_cons, on="Date", how="outer").merge(df_prod, on="Date", how="outer").merge(df_prod_forecast, on="Date", how="outer").merge(df_cons_forecast, on="Date", how="outer")

In [480]:
len(df_scenario1)

52044

In [481]:
df_scenario1.drop_duplicates(subset= "Date", inplace=True)

In [482]:
len(df_scenario1)

51858

In [483]:
df_scenario1.head()

,Date,DAP DE/LU,DAP BE,DAP DK1,DAP FR,DAP NL,DAP CH,DAP IT-N,Total Consumption,Total Production,Reg Production,Conv Production,Total production forecast,Total Consumption forecast,Residual load forecast
0,2019-01-01 00:00:00,28.32,69.49,28.32,51.00,68.92,50.26,51.00,43721.75,53501.25,30060.25,23441.00,51071.0,43647.50,19698.75
1,2019-01-01 01:00:00,10.07,66.58,10.07,46.27,64.98,48.74,46.27,42069.00,52817.25,31767.75,21049.50,51084.0,41692.50,16023.75
2,2019-01-01 02:00:00,-4.08,65.07,-4.08,39.78,60.27,47.24,39.78,40508.00,52169.50,32131.50,20038.00,51513.0,40587.50,13203.50
3,2019-01-01 03:00:00,-9.91,52.17,-9.91,27.87,49.97,36.29,27.87,39682.50,52805.25,33628.00,19177.25,52693.0,40308.25,11298.00
4,2019-01-01 04:00:00,-7.41,47.66,-7.41,23.21,47.66,30.09,22.00,39437.50,54352.75,35395.75,18957.00,53666.0,40659.50,10300.25


In [484]:
df_test = df_scenario1.copy()

In [485]:
df_test.set_index("Date", inplace=True)
full_date_range = pd.date_range(start=df_test.index.min(), end=df_test.index.max(), freq='h')

In [487]:
len(full_date_range)

51864

In [488]:
df_test_reindexed = df_test.reindex(full_date_range)

missing_dates = df_test_reindexed[df_test_reindexed.isnull().any(axis=1)].index
missing_dates

DatetimeIndex(['2019-03-31 02:00:00', '2020-03-29 02:00:00',
               '2021-03-28 02:00:00', '2022-03-27 02:00:00',
               '2023-03-26 02:00:00', '2024-03-31 02:00:00'],
              dtype='datetime64[ns]', freq=None)

In [489]:
df_scenario1.set_index("Date", inplace=True)

for missing_date in missing_dates:
    previous_day = missing_date - pd.Timedelta(hours=24)
    if previous_day in df_scenario1.index:
        df_scenario1.loc[missing_date] = df_scenario1.loc[previous_day]

df_scenario1 = df_scenario1.ffill(limit=24).reset_index(drop=False)

In [490]:
len(df_scenario1)

51864

In [492]:
df_scenario1 = df_scenario1.sort_values(by= "Date").reset_index(drop=True)

In [493]:
df_scenario1.head()

,Date,DAP DE/LU,DAP BE,DAP DK1,DAP FR,DAP NL,DAP CH,DAP IT-N,Total Consumption,Total Production,Reg Production,Conv Production,Total production forecast,Total Consumption forecast,Residual load forecast
0,2019-01-01 00:00:00,28.32,69.49,28.32,51.00,68.92,50.26,51.00,43721.75,53501.25,30060.25,23441.00,51071.0,43647.50,19698.75
1,2019-01-01 01:00:00,10.07,66.58,10.07,46.27,64.98,48.74,46.27,42069.00,52817.25,31767.75,21049.50,51084.0,41692.50,16023.75
2,2019-01-01 02:00:00,-4.08,65.07,-4.08,39.78,60.27,47.24,39.78,40508.00,52169.50,32131.50,20038.00,51513.0,40587.50,13203.50
3,2019-01-01 03:00:00,-9.91,52.17,-9.91,27.87,49.97,36.29,27.87,39682.50,52805.25,33628.00,19177.25,52693.0,40308.25,11298.00
4,2019-01-01 04:00:00,-7.41,47.66,-7.41,23.21,47.66,30.09,22.00,39437.50,54352.75,35395.75,18957.00,53666.0,40659.50,10300.25


In [494]:
df_scenario1

,Date,DAP DE/LU,DAP BE,DAP DK1,DAP FR,DAP NL,DAP CH,DAP IT-N,Total Consumption,Total Production,Reg Production,Conv Production,Total production forecast,Total Consumption forecast,Residual load forecast
0,2019-01-01 00:00:00,28.32,69.49,28.32,51.00,68.92,50.26,51.00,43721.75,53501.25,30060.25,23441.00,51071.0,43647.50,19698.75
1,2019-01-01 01:00:00,10.07,66.58,10.07,46.27,64.98,48.74,46.27,42069.00,52817.25,31767.75,21049.50,51084.0,41692.50,16023.75
2,2019-01-01 02:00:00,-4.08,65.07,-4.08,39.78,60.27,47.24,39.78,40508.00,52169.50,32131.50,20038.00,51513.0,40587.50,13203.50
3,2019-01-01 03:00:00,-9.91,52.17,-9.91,27.87,49.97,36.29,27.87,39682.50,52805.25,33628.00,19177.25,52693.0,40308.25,11298.00
4,2019-01-01 04:00:00,-7.41,47.66,-7.41,23.21,47.66,30.09,22.00,39437.50,54352.75,35395.75,18957.00,53666.0,40659.50,10300.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51859,2024-11-30 19:00:00,131.96,131.96,129.94,131.96,131.96,133.68,144.00,57315.75,53451.50,25936.25,27515.25,52483.0,55192.25,38835.50
51860,2024-11-30 20:00:00,124.51,124.51,116.42,124.51,124.51,126.02,130.00,54442.50,50973.75,24395.75,26578.00,50195.0,52397.50,36690.50
51861,2024-11-30 21:00:00,115.08,115.08,112.18,115.08,115.08,118.67,120.63,52248.25,48334.00,23067.50,25266.50,47663.0,50062.00,35320.50
51862,2024-11-30 22:00:00,114.78,114.78,114.78,114.78,114.78,113.62,114.78,50616.75,46959.50,21469.75,25489.75,45502.0,48596.25,34796.25


### Train test split 

In [519]:
df_forecast = df_scenario1[df_scenario1["Date"] < "2021-01-01"]
df_genmodel = df_scenario1[df_scenario1["Date"].between("2021-01-01", "2023-05-31")]
df_policy = df_scenario1[df_scenario1["Date"] > "2023-05-31"]

In [520]:
df_forecast

,Date,DAP DE/LU,DAP BE,DAP DK1,DAP FR,DAP NL,DAP CH,DAP IT-N,Total Consumption,Total Production,Reg Production,Conv Production,Total production forecast,Total Consumption forecast,Residual load forecast
0,2019-01-01 00:00:00,28.32,69.49,28.32,51.00,68.92,50.26,51.00,43721.75,53501.25,30060.25,23441.00,51071.0,43647.50,19698.75
1,2019-01-01 01:00:00,10.07,66.58,10.07,46.27,64.98,48.74,46.27,42069.00,52817.25,31767.75,21049.50,51084.0,41692.50,16023.75
2,2019-01-01 02:00:00,-4.08,65.07,-4.08,39.78,60.27,47.24,39.78,40508.00,52169.50,32131.50,20038.00,51513.0,40587.50,13203.50
3,2019-01-01 03:00:00,-9.91,52.17,-9.91,27.87,49.97,36.29,27.87,39682.50,52805.25,33628.00,19177.25,52693.0,40308.25,11298.00
4,2019-01-01 04:00:00,-7.41,47.66,-7.41,23.21,47.66,30.09,22.00,39437.50,54352.75,35395.75,18957.00,53666.0,40659.50,10300.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17539,2020-12-31 19:00:00,59.47,61.51,59.47,60.54,57.99,58.49,60.54,55765.00,53186.25,15205.75,37980.50,44945.0,52138.25,43483.00
17540,2020-12-31 20:00:00,56.70,56.79,56.70,56.75,56.64,56.14,56.75,51646.50,51032.25,14239.25,36793.00,45611.0,49651.00,41597.50
17541,2020-12-31 21:00:00,52.44,52.44,52.44,52.44,52.44,52.98,52.44,49203.00,49190.25,13280.50,35909.75,43259.0,47884.00,40427.50
17542,2020-12-31 22:00:00,51.86,51.86,51.86,51.86,51.86,52.31,51.86,47981.00,47542.50,12127.50,35415.00,40388.0,47856.25,40953.75


In [521]:
df_genmodel

,Date,DAP DE/LU,DAP BE,DAP DK1,DAP FR,DAP NL,DAP CH,DAP IT-N,Total Consumption,Total Production,Reg Production,Conv Production,Total production forecast,Total Consumption forecast,Residual load forecast
17544,2021-01-01 00:00:00,50.87,50.87,50.87,50.87,50.87,50.98,50.87,45041.50,42533.75,10412.25,32121.50,43270.0,43520.50,38887.00
17545,2021-01-01 01:00:00,48.19,48.19,48.19,48.19,48.19,45.94,48.19,43255.75,41664.75,10008.00,31656.75,41699.0,41952.25,37833.75
17546,2021-01-01 02:00:00,44.68,44.68,44.68,44.68,44.68,44.53,44.68,41480.75,40942.75,9558.25,31384.50,40723.0,40858.50,37214.50
17547,2021-01-01 03:00:00,42.92,42.92,42.92,42.92,42.92,41.01,42.92,40658.75,40528.50,9148.50,31380.00,40047.0,40490.25,37224.25
17548,2021-01-01 04:00:00,40.39,40.39,40.39,40.39,40.39,39.23,40.39,40633.50,40046.25,8811.25,31235.00,39856.0,40190.50,37196.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38660,2023-05-30 20:00:00,154.56,119.49,154.56,105.08,132.10,119.93,144.99,53041.25,40667.00,15982.75,24684.25,43015.0,55215.50,46953.25
38661,2023-05-30 21:00:00,125.77,102.40,125.77,99.58,94.55,110.79,120.66,50935.00,40435.00,16509.00,23926.00,41833.0,53066.75,44246.75
38662,2023-05-30 22:00:00,97.31,96.13,97.31,95.39,97.17,95.32,98.52,48465.50,38993.75,18620.00,20373.75,40944.0,50353.75,40370.50
38663,2023-05-30 23:00:00,89.65,89.65,89.65,89.65,89.65,87.27,89.65,44302.00,36345.25,19644.50,16700.75,37295.0,46803.50,36277.00


In [522]:
df_policy

,Date,DAP DE/LU,DAP BE,DAP DK1,DAP FR,DAP NL,DAP CH,DAP IT-N,Total Consumption,Total Production,Reg Production,Conv Production,Total production forecast,Total Consumption forecast,Residual load forecast
38665,2023-05-31 01:00:00,82.17,73.41,82.17,68.40,81.36,83.48,80.60,39973.50,32510.00,18652.00,13858.00,33754.0,40623.50,30260.75
38666,2023-05-31 02:00:00,74.20,74.20,74.20,74.20,74.20,82.01,78.50,39257.75,30964.50,17488.25,13476.25,33165.0,39655.25,30016.25
38667,2023-05-31 03:00:00,80.02,62.64,78.10,52.48,78.09,80.99,80.00,39660.50,30675.50,16523.00,14152.50,32326.0,39763.00,30991.00
38668,2023-05-31 04:00:00,85.26,61.61,82.24,47.73,82.24,83.48,79.89,40431.00,30409.00,15840.75,14568.25,32397.0,41167.00,33110.50
38669,2023-05-31 05:00:00,91.98,72.70,88.92,61.12,88.92,86.64,86.52,43216.25,31146.50,15738.50,15408.00,33394.0,44149.00,36305.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51859,2024-11-30 19:00:00,131.96,131.96,129.94,131.96,131.96,133.68,144.00,57315.75,53451.50,25936.25,27515.25,52483.0,55192.25,38835.50
51860,2024-11-30 20:00:00,124.51,124.51,116.42,124.51,124.51,126.02,130.00,54442.50,50973.75,24395.75,26578.00,50195.0,52397.50,36690.50
51861,2024-11-30 21:00:00,115.08,115.08,112.18,115.08,115.08,118.67,120.63,52248.25,48334.00,23067.50,25266.50,47663.0,50062.00,35320.50
51862,2024-11-30 22:00:00,114.78,114.78,114.78,114.78,114.78,113.62,114.78,50616.75,46959.50,21469.75,25489.75,45502.0,48596.25,34796.25


In [523]:
17544 + 21121 + 13199

51864

In [ ]:
51864

In [572]:
df_forecast.to_csv("/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/Data/forecast.csv", index=False)
df_genmodel.to_csv("/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/Data/genmodel.csv", index=False)
df_policy.to_csv("/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/Data/policy.csv", index=False)